# Implementación de RAG usando LangChain
Diseñar e implementar un sistema básico de RAG (Retrieval-Augmented Generation) usando LangChain que pueda responder preguntas utilizando información contenida en un conjunto de documentos locales.

## **Parte 1: Preparación del entorno**

- Instala las librerías necesarias: langchain, faiss-cpu, openai, tiktoken

- Registra una cuenta en OpenAI (o usa un modelo local como llama-cpp o ollama si lo prefieres).

### Instalar las librerías necesarias

In [49]:
!pip install langchain
!pip install langchain-community
!pip install langchain-core
!pip install -U langchain-openai
!pip install langchain openai weaviate-client
!pip install faiss-cpu
!pip install -U langchain-openai

### Registra una cuenta en OpenAI

In [50]:
from google.colab import userdata
import requests

# Obtener secrets
OPENAI_API_KEY = userdata.get("open-ai-api-key")

## **Parte 2: Construcción del sistema RAG**
1. Carga de documentos: Usa un conjunto de documentos .txt,  .html, .pdf sobre un tema específico (ej. artículos de Wikipedia, documentos científicos o manuales técnicos).

  Puedes usar textos de dominio público de:
  - 📚 Project GutenbergLinks
  - 📄 arXiv

2. División en fragmentos: Usa CharacterTextSplitter para dividir los documentos en chunks de 500 caracteres.

### Carga de documentos

In [51]:
# Obtener libros de la Biblia disponibles de Project Gutenberg en español
url_libros = [
    "https://gkpy.net/ebooks/12500.txt", # Buena Nueva de acuerdo a Mateo
    "https://gkpy.net/ebooks/12501.txt", # Buena Nueva de acuerdo a Marcos
    "https://gkpy.net/ebooks/12502.txt", # Buena Nueva de acuerdo a Lucas
    "https://gkpy.net/ebooks/12503.txt", # Buena Nueva de acuerdo a Juan
]

In [52]:
def descargar_texto(url):
    respuesta = requests.get(url)
    respuesta.encoding = "utf-8"
    return respuesta.text

In [53]:
# Descargar y guardar los textos en una lista
documentos = []

for i, url in enumerate(url_libros):
    texto = descargar_texto(url)
    documentos.append(texto)
    print(f"✅ Libro {i+1} descargado - {len(texto)} caracteres")

✅ Libro 1 descargado - 192255 caracteres
✅ Libro 2 descargado - 124704 caracteres
✅ Libro 3 descargado - 200199 caracteres
✅ Libro 4 descargado - 161364 caracteres


### Limpiar los documentos

In [54]:
def limpiar_texto_gutenberg(texto, start_markers, end_markers):
    inicio_idx = 0
    fin_idx = len(texto)

    for marker in start_markers:
        idx = texto.find(marker)
        if idx != -1:
            inicio_idx = max(inicio_idx, idx + len(marker))  # cortamos después del marcador

    for marker in end_markers:
        idx = texto.find(marker)
        if idx != -1:
            fin_idx = min(fin_idx, idx)

    return texto[inicio_idx:fin_idx].strip()

In [55]:
# Basado en https://github.com/pgcorpus/gutenberg/blob/master/src/cleanup.py

TEXT_START_MARKERS = frozenset((
    "*END*THE SMALL PRINT",
    "*** START OF THE PROJECT GUTENBERG",
    "*** START OF THIS PROJECT GUTENBERG",
    "This etext was prepared by",
    "E-text prepared by",
    "Produced by",
    "Distributed Proofreading Team",
    "Proofreading Team at http://www.pgdp.net",
    "http://gallica.bnf.fr)",
    "      http://archive.org/details/",
    "http://www.pgdp.net",
    "by The Internet Archive)",
    "by The Internet Archive/Canadian Libraries",
    "by The Internet Archive/American Libraries",
    "public domain material from the Internet Archive",
    "Internet Archive)",
    "Internet Archive/Canadian Libraries",
    "Internet Archive/American Libraries",
    "material from the Google Print project",
    "*END THE SMALL PRINT",
    "***START OF THE PROJECT GUTENBERG",
    "This etext was produced by",
    "*** START OF THE COPYRIGHTED",
    "The Project Gutenberg",
    "http://gutenberg.spiegel.de/ erreichbar.",
    "Project Runeberg publishes",
    "Beginning of this Project Gutenberg",
    "Project Gutenberg Online Distributed",
    "Gutenberg Online Distributed",
    "the Project Gutenberg Online Distributed",
    "Project Gutenberg TEI",
    "This eBook was prepared by",
    "http://gutenberg2000.de erreichbar.",
    "This Etext was prepared by",
    "This Project Gutenberg Etext was prepared by",
    "Gutenberg Distributed Proofreaders",
    "Project Gutenberg Distributed Proofreaders",
    "the Project Gutenberg Online Distributed Proofreading Team",
    "**The Project Gutenberg",
    "*SMALL PRINT!",
    "More information about this book is at the top of this file.",
    "tells you about restrictions in how the file may be used.",
    "l'authorization à les utilizer pour preparer ce texte.",
    "of the etext through OCR.",
    "*****These eBooks Were Prepared By Thousands of Volunteers!*****",
    "We need your donations more than ever!",
    " *** START OF THIS PROJECT GUTENBERG",
    "****     SMALL PRINT!",
    '["Small Print" V.',
    '      (http://www.ibiblio.org/gutenberg/',
    'and the Project Gutenberg Online Distributed Proofreading Team',
    'Mary Meehan, and the Project Gutenberg Online Distributed Proofreading',
    '                this Project Gutenberg edition.',
))


TEXT_END_MARKERS = frozenset((
    "*** END OF THE PROJECT GUTENBERG",
    "*** END OF THIS PROJECT GUTENBERG",
    "***END OF THE PROJECT GUTENBERG",
    "End of the Project Gutenberg",
    "End of The Project Gutenberg",
    "Ende dieses Project Gutenberg",
    "by Project Gutenberg",
    "End of Project Gutenberg",
    "End of this Project Gutenberg",
    "Ende dieses Projekt Gutenberg",
    "        ***END OF THE PROJECT GUTENBERG",
    "*** END OF THE COPYRIGHTED",
    "End of this is COPYRIGHTED",
    "Ende dieses Etextes ",
    "Ende dieses Project Gutenber",
    "Ende diese Project Gutenberg",
    "**This is a COPYRIGHTED Project Gutenberg Etext, Details Above**",
    "Fin de Project Gutenberg",
    "The Project Gutenberg Etext of ",
    "Ce document fut presente en lecture",
    "Ce document fut présenté en lecture",
    "More information about this book is at the top of this file.",
    "We need your donations more than ever!",
    "END OF PROJECT GUTENBERG",
    " End of the Project Gutenberg",
    " *** END OF THIS PROJECT GUTENBERG",
))

LEGALESE_START_MARKERS = frozenset(("<<THIS ELECTRONIC VERSION OF",))
LEGALESE_END_MARKERS = frozenset(("SERVICE THAT CHARGES FOR DOWNLOAD",))

# Limpiar todos los documentos descargados y eliminar líneas vacías
documentos_limpios = []

for texto in documentos:
    limpio = limpiar_texto_gutenberg(texto, TEXT_START_MARKERS, TEXT_END_MARKERS)
    # Eliminar líneas vacías
    limpio_sin_lineas_vacias = "\n".join([linea for linea in limpio.splitlines() if linea.strip() != ""])
    documentos_limpios.append(limpio_sin_lineas_vacias)

# Verificamos la limpieza
print("LIMPIO:", documentos_limpios[0][:100])
print("ORIGINAL:", documentos[0][:100])

LIMPIO: EBOOK BUENA NUEVA DE ACUERDO A MATEO: TRADUCCIÓN DE DOMINIO PÚBLICO ABIERTA A MEJORAS ***
Esta tradu
ORIGINAL: ﻿The Project Gutenberg eBook of Buena Nueva de acuerdo a Mateo: Traducción de dominio público abiert


### División en fragmentos

In [127]:
from langchain.text_splitter import CharacterTextSplitter

# Crear el splitter: 500 caracteres por chunk
splitter = CharacterTextSplitter(
    separator="",
    chunk_size=500,
    chunk_overlap=20,
)

# Dividir cada documento en chunks
all_chunks = []
for texto in documentos_limpios:
    chunks = splitter.split_text(texto)
    all_chunks.extend(chunks)

print(f"Total de chunks generados: {len(all_chunks)}")
print("Ejemplo de chunk:")
print(all_chunks[10])
print(len(documentos_limpios))

Total de chunks generados: 1210
Ejemplo de chunk:
«Dios está con nosotros.» 
001:024 José despertó de su sueño, e hizo lo que el ángel del Señor le
        ordenó, y tomó a su esposa consigo;
001:025 y no la conoció[7] hasta que ella dio a luz a su primer hijo.
        Él[8] lo llamó Jesús.
        2
002:001 Cuando Jesús nació en Belén de Judea, en los días que Herodes
        era rey, ocurrió, que desde el este vinieron hombres sabios[9]
        hacia Jerusalén, diciendo,
002:002 «¿Dónde está aquel que nace como Rey de los Judíos? Porque
4


### Creación del índice semántico

  Usa FAISS para construir un índice vectorial a partir de los chunks usando embeddings como OpenAIEmbeddings.

In [128]:
from langchain_core.documents import Document

document_chunks = [Document(page_content=chunk) for chunk in all_chunks]

In [129]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# Crear embeddings con OpenAI
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
# Crear el índice vectorial
vectorstore = FAISS.from_documents(document_chunks, embeddings)

### Configuración del LLM

Usa OpenAIChat o ChatOpenAI si estás usando GPT-3.5 o (LLamaCpp o ollama) si estás trabajando localmente.
### Construcción del RAG chain

Crea un RetrievalQA chain que combine el retriever (índice FAISS) con el modelo generador (LLM).

In [130]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.vectorstores import FAISS
from langchain.prompts import PromptTemplate

In [131]:
llm = ChatOpenAI(api_key=OPENAI_API_KEY, model="gpt-3.5-turbo")

retriever = vectorstore.as_retriever()


template = """
Responde a la siguiente pregunta solamente utilizando el contexto proporcionado.
No inventes información. Si no hay suficiente información en el contexto, responde "No sé".

Tu respuesta debe ser clara, completa y comprensible, incluso si es breve.
Usa un lenguaje natural y evita ser excesivamente vago.
Contexto:
{context}

Pregunta:
{question}
"""

custom_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={
        "prompt": custom_prompt
    }
)

### Evaluación del sistema
- Formula al menos 5 preguntas que solo puedan responderse con los documentos cargados.
- Evalúa la calidad y precisión de las respuestas.

In [132]:
query = "¿Jesús tenía hermanos?"
respuesta = qa_chain.invoke(query)

In [133]:
respuesta

{'query': '¿Jesús tenía hermanos?',
 'result': 'Sí.',
 'source_documents': [Document(id='e6352443-ca28-4bba-8028-0a604681897b', metadata={}, page_content='y mis hermanos!\n012:050 Pues cualquiera que haga la voluntad de mi Padre que está en\n        el cielo, es mi hermano, mi hermana y madre.»\n        13\n013:001 En ese día Jesús salió de la casa, y se sentó a la orilla del\n        lago.\n013:002 Grandes multitudes se reunieron con Él, así que Jesús entró en\n        un bote, y se sentó y la multitud se quedó en la playa.\n013:003 Les habló muchas cosas en parábolas, diciendo, «Observen, un\n        campesino salió a sembrar.\n013:004 Mientras sembraba,'),
  Document(id='75254d1a-c4d5-4469-a90b-10fadef1febf', metadata={}, page_content='us hermanos llegaron donde Él estaba, pero no\n        pudieron acercarse debido a la multitud.\n008:020 Alguien le dijo, «Tu madre y tus hermanos están afuera,\n        deseando verte.»\n008:021 Pero Él les contestó, «Mi madre y mis hermanos son quie

Correcto, Jesús tenía hermanos.

<hr/>

In [134]:
query = "¿Quién era Jesús?"
respuesta = qa_chain.invoke(query)

In [135]:
respuesta

{'query': '¿Quién era Jesús?',
 'result': 'Jesús era el Cristo, el Hijo del Dios viviente.',
 'source_documents': [Document(id='a750150d-2067-49ed-8466-ecaaa180fda6', metadata={}, page_content='los fariseos y los saduceos.\n016:013 Cuando Jesús entró a las región de Cesarea de Filipo les\n        preguntó a sus discípulos, «¿Quién dicen los hombres que soy\n        yo, el Hijo del Hombre[137]?»\n016:014 Ellos dijeron, «Algunos dicen Juan el Bautista, algunos Elías\n        y otros que Jeremías o alguno de los profetas.»\n016:015 Él les dijo, «¿Pero quién dicen ustedes que soy yo?»\n016:016 Simón Pedro respondió, «Tu eres el Cristo, el Hijo del Dios\n        viviente.»\n016:017 Jesús le contestó'),
  Document(id='fadce62a-e7dd-49bb-9970-f214e18271bb', metadata={}, page_content='es por señas [153], Simón Pedro le dijo, «Dinos quien es\n        aquel de quien Él habla.»\n013:025 Inclinándose de nuevo como estaba, sobre el pecho de\n        Jesús[154] le preguntó, «Señor, ¿Quién es?»\n013:

La respuesta es correxta.

<hr />

In [137]:
query = "¿Qué dijo Jesús a la mujer acusada de adulterio?"
respuesta = qa_chain.invoke(query)

In [138]:
respuesta

{'query': '¿Qué dijo Jesús a la mujer acusada de adulterio?',
 'result': 'Jesús dijo: "Yo tampoco te condeno. Ve por tu camino. Desde ahora no peques más."',
 'source_documents': [Document(id='20be8456-3b36-4210-8222-de0dbe555a2a', metadata={}, page_content='mo. Jesús fue dejado solo con la mujer, la cual se\n        encontraba aún en la mitad.\n008:010 Jesús levantándose la miró y dijo, «Mujer, ¿Dónde están los\n        que te acusan?[97] ¿Nadie te condenó?»\n008:011 Ella dijo, «Nadie Señor.»\n        Jesús dijo, «Yo tampoco te condeno. Ve por tu camino. Desde\n        ahora no peques más.»\n008:012 En otra ocasión Jesús les habló diciéndoles[98], «Yo soy la\n        luz del mundo. El que me siga no caminará en la oscuridad,\n        tendrá la luz de la vid'),
  Document(id='a9905434-f3f7-44b7-ad7d-89d63a34310e', metadata={}, page_content='multitud, y preguntó, «¿Quien tocó mi ropa?»\n005:031 Sus discípulos le dijeron, «Mira la multitud presionándote,\n        Como dices, `¿Quien me t

La respuesta es correcta. Aunque no sea lo primero que dijo Jesús a la mujer, pero creo que es lo más relevante.

<hr/>

In [139]:
query = "¿De quién hablaba Jesús cuando dijo: '¿No los escogí a los doce? Y uno de ustedes un demonio'"
respuesta = qa_chain.invoke(query)

In [140]:
respuesta

{'query': "¿De quién hablaba Jesús cuando dijo: '¿No los escogí a los doce? Y uno de ustedes un demonio'",
 'result': 'Jesús hablaba de Judas, el hijo de Simón Iscariote.',
 'source_documents': [Document(id='0436bc66-3f67-4ed7-9e5b-11eb79954230', metadata={}, page_content='ren irse,\n        ¿O si quieren?»\n006:068 Simón Pedro le contesto, «Señor, ¿A quien iríamos? Tu tienes\n        las palabras de la vida eterna.\n006:069 Hemos llegado a creer y sabemos que tu eres Dios\n        bendito[82].»\n006:070 Jesús les contesto, «¿No los escogí a los doce? Y uno de\n        ustedes un demonio»\n006:071 Él hablaba de Judas, el hijo de Simón Iscariote, porque este\n        era el que lo traicionaría, y era uno de los doce.\n        7\n007:001 Después de estas cosas, Jesús anduvo po'),
  Document(id='fadce62a-e7dd-49bb-9970-f214e18271bb', metadata={}, page_content='es por señas [153], Simón Pedro le dijo, «Dinos quien es\n        aquel de quien Él habla.»\n013:025 Inclinándose de nuevo como es

Esta respuesta es correcta.

<hr />

In [141]:
query = "¿Cómo se llama la festividad en la que se come pan sin levadura?"
respuesta = qa_chain.invoke(query)

In [142]:
respuesta

{'query': '¿Cómo se llama la festividad en la que se come pan sin levadura?',
 'result': 'La festividad en la que se come pan sin levadura se llama la Pascua.',
 'source_documents': [Document(id='f31136bc-4851-4334-a1bd-2e106072d42e', metadata={}, page_content='l primer día de pan sin levadura, cuando ofrecían la Pascua,\n        sus discípulos le preguntaron, «¿Donde quieres que vayamos a\n        preparar la cena de Pascua?»\n014:013 Él envió a dos de sus discípulos, y les dijo, «Vayan a la\n        ciudad, allí encontrarán un hombre cargando un jarro de agua.\n        Síganlo,\n014:014 y donde él entre, díganle al dueño de la casa, `Él Maestro\n        dice «¿ Donde está el cuarto de invitados, donde podré hacer\n        la cena de Pascua con mis discípulos'),
  Document(id='ce54c0f1-7dda-4168-bac8-674809faa809', metadata={}, page_content='mprano en la mañana al templo\n        para escucharlo.\n        22\n022:001 Ahora estaba cerca la festividad en la que se come pan sin\n        

La respuesta es correcta. Pero en el texto utilizado como source, no se menciona.

<hr/>

In [143]:
query = "¿Con qué compararemos el Reino de Dios?"
respuesta = qa_chain.invoke(query)

In [144]:
respuesta

{'query': '¿Con qué compararemos el Reino de Dios?',
 'result': 'Con un grano de semilla de mostaza y con levadura.',
 'source_documents': [Document(id='26092087-3f2a-4fd6-a583-6a59aa48bef2', metadata={}, page_content='do hechas por Él.\n013:018 Dijo, «¿Cómo qué es el Reino de Dios? ¿Con qué lo compararé?\n013:019 Es como un grano de la semilla de mostaza, que un hombre tomó,\n        y puso en su propio jardín. Creció y se convirtió en un gran\n        árbol, y los pájaros del cielos descansaban en sus ramas.»\n013:020 Nuevamente dijo, «¿Con qué compararé el Reino de Dios?\n013:021 Es como levadura, que una mujer toma y esconde en tres\n        medidas[116] de harina, hasta que toda queda impregnada.»\n013:022 Siguió'),
  Document(id='bf7b970a-aa19-43c7-9aa9-e94c30a89140', metadata={}, page_content='s impidan pues el Reino de Dios pertenece a los que\n        son como ellos.\n018:017 Con seguridad les digo, quien no recibe el Reino de Dios como\n        un niño, no entrará en él de ni

Esta respuesta es correcta.

<hr/>

In [145]:
query = "¿Qué es como una semilla de mostaza?"
respuesta = qa_chain.invoke(query)

In [146]:
respuesta

{'query': '¿Qué es como una semilla de mostaza?',
 'result': 'Un grano de mostaza es como el Reino de Dios.',
 'source_documents': [Document(id='26092087-3f2a-4fd6-a583-6a59aa48bef2', metadata={}, page_content='do hechas por Él.\n013:018 Dijo, «¿Cómo qué es el Reino de Dios? ¿Con qué lo compararé?\n013:019 Es como un grano de la semilla de mostaza, que un hombre tomó,\n        y puso en su propio jardín. Creció y se convirtió en un gran\n        árbol, y los pájaros del cielos descansaban en sus ramas.»\n013:020 Nuevamente dijo, «¿Con qué compararé el Reino de Dios?\n013:021 Es como levadura, que una mujer toma y esconde en tres\n        medidas[116] de harina, hasta que toda queda impregnada.»\n013:022 Siguió'),
  Document(id='95b1fdd2-5b57-4d3e-9a8f-821ca259c4fb', metadata={}, page_content='el trigo en mi granero.»´»\n013:031 Él les ofreció otra parábola, diciendo, «El Reino de Dios es\n        como un grano de la semilla de mostaza, que un hombre tomó, y\n        sembró en su campo;

Esta respuesta es correcta.

<hr/>